In [ ]:
# ==============================================================================
# Adversarial Robustness Evaluation — English track
# MODIFIED: the 'Proposed Method' slot now loads the NEW head from
# DeBERTaV3_SupCon_MLP_AttnPool (SupCon encoder + MLPHead sentence head +
# AttnPool doc head whose classifier is the shared MLPHead class). Changes vs
# the previous version: checkpoint folder/file names, state_dict keys
# (attn_doc_head_state_dict), MLPHead classifier (classifier.net.*),
# config keys (config_*.json, s3_best_val_f1), and sentence pipeline
# (normalize whole doc -> split -> >=8 words) exactly as in training.
# (see chat, 2026-10-22): REBUILD of the original "FourMethods" robustness
# script (Direct/Retrieval/GNN/Fusion-Avg/Fusion-Meta, loaded from a single
# four_methods_model.pt checkpoint with FAISS + torch_geometric). That whole
# architecture family is REMOVED. In its place, EXACTLY the same seven
# methods used throughout the rest of this project's cross-domain/
# cross-generator tests:
#   - The six standard DOCUMENT-LEVEL fine-tuned classifiers (BERT, RoBERTa,
#     ELECTRA, DeBERTa-v3, XLNet, DistilBERT) — each document fed as ONE
#     sequence (MAX_LENGTH=512), exactly like
#     external_cross_generator_test_six_doclevel_plus_attnpool.py.
#   - Attention-Pooling (Doc) — frozen SupCon (contrastive-learning) encoder
#     + the trained AttnPoolDocHead: sentence embeddings combined via a
#     learned masked-softmax attention-weighted sum, instead of a plain
#     mean or the old Dual-SVM/GNN/retrieval machinery.
# No FAISS, no torch_geometric, no retrieval index, no GNN, no meta-
# classifier anywhere in this script.
#
# HEATMAP COLORS (see chat, 2026-10-22): the old rainbow "VIVID" colormap
# (red->orange->yellow->green->blue) is REPLACED with this project's
# Q1-journal theme — a single-hue SEQUENTIAL blue ramp for detection rate
# (light = low, dark = high) and a matching sequential RED ramp for the
# degradation/drop heatmap (light = no drop, dark = large drop), both with
# the same per-cell luminance-adaptive text color used in every other
# heatmap in this project, instead of plain black/white annotation.
#
# TABLES (see chat, 2026-10-22; extended 2026-10-23 with two more
# methods): every table now has 9 method columns (BERT, RoBERTa,
# ELECTRA, DeBERTa-v3, XLNet, DistilBERT, Attention-Pooling (Doc),
# DeBERTa-v3 (SupCon+MLP+AttnPool), RoBERTa (SupCon+MLP+AttnPool))
# instead of 5, each scored against ITS OWN validated threshold
# (THRESHOLDS_DOC below) instead of a single shared 0.5 cutoff, and a new
# small Threshold reference table is printed/saved so every table's
# cutoffs are documented in one place.
# ==============================================================================
# قبل التشغيل بخلية منفصلة:
#   !pip install -q -U transformers sentencepiece

import os, re, shutil, unicodedata, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from tqdm.auto import tqdm

from google.colab import drive, files
drive.mount("/content/drive", force_remount=True)

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("جهاز التشغيل:", DEVICE)

# ==============================================================================
# 0. Q1-JOURNAL PUBLICATION PLOT THEME — SAME helpers used across this
# project's cross-domain/cross-generator test scripts (sequential heatmap,
# dynamic per-cell text color, panel letters, white-halo bar labels, 300
# DPI), plus a new sequential RED ramp mirroring BLUE_ONLY_CMAP's
# construction for the degradation/drop heatmap.
# ==============================================================================
ATTN_KEY = "Proposed Method"   # renamed from "Attention-Pooling (Doc)" (see chat, 2026-10-23)

METHOD_ORDER = ["BERT", "RoBERTa", "ELECTRA", "DeBERTa-v3", "XLNet", "DistilBERT", ATTN_KEY]
METHOD_COLORS = ["#2E5EAA", "#22A6B3", "#4C72B0", "#C0392B", "#8E44AD", "#7F8C8D", "#D35400"]
CM_CMAPS = ["Blues", "GnBu", "PuBu", "Reds", "Purples", "Greys", "OrRd"]

BLUE_ONLY_CMAP = LinearSegmentedColormap.from_list(
    "light_to_dark_blue",
    ["#F5F9FD", "#D6E7F7", "#B0D0EF", "#82B4E3", "#4F92D4", "#2E72B8", "#1A5596"],
    N=256,
)
RED_ONLY_CMAP = LinearSegmentedColormap.from_list(
    "light_to_dark_red",
    ["#FDF5F5", "#F7D6D6", "#EFB0B0", "#E38282", "#D44F4F", "#B82E2E", "#961A1A"],
    N=256,
)
GRID_COLOR = "#DDE3EC"
AXIS_COLOR = "#2B2B2B"
BASELINE_COLOR = "#2E72B8"   # matches BLUE_ONLY_CMAP's mid-dark blue
ATTACK_COLOR = "#B82E2E"     # matches RED_ONLY_CMAP's mid-dark red

def heatmap_dynamic_text(ax, data, vmin, vmax, cmap, fmt="{:.3f}", fontsize=11):
    span = max(vmax - vmin, 1e-9)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data.iloc[i, j] if hasattr(data, "iloc") else data[i][j]
            norm = (float(val) - vmin) / span
            r, g, b, _ = cmap(np.clip(norm, 0, 1))
            luminance = 0.299 * r + 0.587 * g + 0.114 * b
            text_color = "white" if luminance < 0.55 else "#1A1A1A"
            label = fmt.format(val) if not isinstance(val, str) else val
            ax.text(j + 0.5, i + 0.5, label, ha="center", va="center",
                    fontsize=fontsize, fontweight="bold", color=text_color)

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": AXIS_COLOR, "axes.linewidth": 1.1,
    "axes.grid": False, "axes.axisbelow": True,
    "grid.color": GRID_COLOR, "grid.linewidth": 0.7, "grid.alpha": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11.5, "font.family": "DejaVu Serif",
    "axes.titleweight": "bold", "axes.titlesize": 13,
    "axes.labelsize": 11.5, "axes.labelweight": "medium",
    "legend.frameon": False, "legend.fontsize": 9.5,
    "xtick.labelsize": 10, "ytick.labelsize": 10,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "savefig.facecolor": "white",
})
BAR_W = 0.5

def style_axis(ax, title=None, xlabel=None, ylabel=None):
    if title:  ax.set_title(title, fontweight="bold", pad=10)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", linestyle="-", linewidth=0.7, color=GRID_COLOR, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(AXIS_COLOR)
        ax.spines[spine].set_linewidth(1.0)
    return ax

def add_panel_label(ax, label):
    ax.text(-0.06, 1.08, label, transform=ax.transAxes, fontsize=14,
             fontweight="bold", va="top", ha="left", color=AXIS_COLOR)

def label_bars(ax, bars, values, fmt="{:.4f}", dy=0.012, fontsize=8.5):
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + dy, fmt.format(v), ha="center",
                 fontsize=fontsize, fontweight="bold", color=AXIS_COLOR,
                 path_effects=[pe.withStroke(linewidth=2.2, foreground="white")])

# ==============================================================================
# 1. SHARED TEXT PIPELINE — IDENTICAL to
# external_cross_generator_test_six_doclevel_plus_attnpool.py. The six
# classifiers take the whole normalized document as ONE sequence; only
# Attention-Pooling needs this sentence-splitting pipeline.
# ==============================================================================
_SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')
MIN_SENT_WORDS = 8

def sent_tokenize_en(text):
    if not isinstance(text, str):
        return []
    text = re.sub(r'\s*\n+\s*', ' ', text.strip())
    if not text:
        return []
    return [p.strip() for p in _SENT_SPLIT_RE.split(text) if p.strip()]

_INVISIBLE_CHARS_RE = re.compile('[​‌‍‎‏‪‫‬‭‮﻿]')
_CITATION_RE = re.compile(
    r'\[\d+(?:\s*,\s*\d+)*\]'
    r'|\([^()]{0,60}\d{4}[^()]{0,25}\)'
)
_URL_RE   = re.compile(r'(?:https?://|www\.)\S+')
_EMAIL_RE = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')
_EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "]+", flags=re.UNICODE
)
PUNCTUATION = r";:\"'`“”‘’«»\-–—/\\|@#\$%\^&\*_=~<>\(\)\[\]\{\}"
_NUMERIC_COMMA_SAFE_RE = re.compile(r'(?<!\d),|,(?!\d)')
_SPACE_BEFORE_SENT_END_RE = re.compile(r'\s+([.!?])')

_CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'ve": " have", "'m": " am",
}
_CONTRACTIONS_RE = re.compile("|".join(re.escape(k) for k in _CONTRACTIONS.keys()))

def normalize_english_text(text):
    if not isinstance(text, str):
        return text
    text = _INVISIBLE_CHARS_RE.sub('', text)
    text = _CITATION_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = _EMOJI_RE.sub(' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = _CONTRACTIONS_RE.sub(lambda m: _CONTRACTIONS[m.group(0)], text)
    text = _NUMERIC_COMMA_SAFE_RE.sub(" ", text)
    text = re.sub(rf"[{PUNCTUATION}]", " ", text)
    text = _SPACE_BEFORE_SENT_END_RE.sub(r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_english_text_attnpool(text):
    """Normalization used by the NEW AttnPool training script: identical to
    normalize_english_text but WITHOUT the final 'glue . ! ? to previous
    word' step (that step is not part of the training pipeline of this head)."""
    if not isinstance(text, str):
        return text
    text = _INVISIBLE_CHARS_RE.sub('', text)
    text = _CITATION_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = _EMOJI_RE.sub(' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = _CONTRACTIONS_RE.sub(lambda m: _CONTRACTIONS[m.group(0)], text)
    text = _NUMERIC_COMMA_SAFE_RE.sub(" ", text)
    text = re.sub(rf"[{PUNCTUATION}]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def doc_to_sentences_for_pooling(raw_text):
    """MODIFIED — mirrors the training script exactly: normalize the WHOLE
    document first, then split into sentences, keep those with >=
    MIN_SENT_WORDS words, and fall back to the whole normalized text if none
    qualify. The six classifiers do NOT use this."""
    norm = normalize_english_text_attnpool(raw_text)
    if not isinstance(norm, str) or not norm.strip():
        return []
    sents = [s for s in sent_tokenize_en(norm) if len(s.split()) >= MIN_SENT_WORDS]
    if not sents:
        sents = [norm]
    return sents

# ==============================================================================
# 2. CONFIG — عدّل المسارات هنا فقط
# ==============================================================================
PROJECT_DIR = "/content/drive/MyDrive/english_ai_detector_project"

CLASSIFIER_PATHS = {
    "BERT":        f"{PROJECT_DIR}/BERT_DocumentLevel_FineTune/model_final_bert_doclevel",
    "RoBERTa":     f"{PROJECT_DIR}/RoBERTa_DocumentLevel_FineTune/model_final_roberta_doclevel",
    "ELECTRA":     f"{PROJECT_DIR}/ELECTRA_DocumentLevel_FineTune/model_final_electra_doclevel",
    "DeBERTa-v3":  f"{PROJECT_DIR}/DeBERTaV3_DocumentLevel_FineTune/model_final_deberta_v3_doclevel",
    "XLNet":       f"{PROJECT_DIR}/XLNet_DocumentLevel_FineTune/model_final_xlnet_doclevel",
    "DistilBERT":  f"{PROJECT_DIR}/DistilBERT_DocumentLevel_FineTune/model_final_distilbert_doclevel",
}

THRESHOLDS_DOC = {
    "BERT":        0.57,
    "RoBERTa":     0.55,
    "ELECTRA":     0.58,
    "DeBERTa-v3":  0.46,
    "XLNet":       0.52,
    "DistilBERT":  0.56,
    # ATTN_KEY's threshold is NOT hardcoded here — it is read directly from
    # its own saved head_config.json (th_doc) in Section 4 below.
}

# Frozen SupCon (contrastive-learning) encoder — Attention-Pooling (Doc) was
# trained on top of this exact encoder (loaded, not retrained).
M1_DRIVE_PATH = f"{PROJECT_DIR}/DeBERTaV3_SupCon_MLP_AttnPool"   # MODIFIED: new encoder + head folder
NEW_SUFFIX = "mlp_attnpool_deberta"                               # MODIFIED: suffix used by the training script
M1_ENCODER_NAME = "microsoft/deberta-v3-base"
M1_PROJ_HID, M1_PROJ_DIM = 512, 128
M1_MAX_LEN = 128   # per-sentence encoding length, same as that method's own scripts

ATTNPOOL_HEAD_DRIVE_PATH = M1_DRIVE_PATH   # MODIFIED: head is saved in the same folder as the encoder
ATTN_HIDDEN = 256       # must match that script's CFG["attn_hidden"]
HEAD_DROPOUT = 0.3      # unused at inference (eval mode), kept for class signature

ATTACKS_FILE = "/content/kufa_ai_generated_attacked.xlsx"
EVAL_SAMPLE_SIZE = 2000     # None = يشغل على كل الصفوف | رقم = عينة أسرع للتجربة
OUTPUT_DIR = "/content/robustness_eval_results"
DRIVE_EVAL_OUT = os.path.join(PROJECT_DIR, "robustness_eval_six_doclevel_plus_attnpool")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_EVAL_OUT, exist_ok=True)

DOC_MAX_LEN = 512       # the six document-level classifiers — whole document as ONE sequence
DOC_INFER_BATCH = 16    # kept modest (vs. 32 at training) since MAX_LENGTH=512 sequences are long

ATTACK_COLUMNS = {
    "text_original":           "Baseline (No Attack)",
    "attack_synonym":          "Synonym Replacement",
    "attack_typo":             "Typo Injection",
    "attack_human_ai_mix":     "Human-AI Mix",
    "attack_translation":      "Back-Translation",
    "quillbot humanizer":      "QuillBot Humanizer",
    "grammarly humanizer":     "Grammarly Humanizer",
}

print("=" * 70)
print("  Adversarial Robustness Evaluation — 6 document-level classifiers + Proposed Method")
print("=" * 70)

# ==============================================================================
# 3. LOAD THE SIX DOCUMENT-LEVEL CLASSIFIERS
# ==============================================================================
print("\n" + "─" * 70)
print("  Loading the six DOCUMENT-LEVEL fine-tuned classifiers ...")
CLASSIFIERS = {}
for name, path in CLASSIFIER_PATHS.items():
    print(f"    {name:12s} <- {path}")
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE).eval()
    CLASSIFIERS[name] = (tok, mdl)
print("  All six document-level classifiers loaded.")

# ==============================================================================
# 4. LOAD Attention-Pooling (Doc) — frozen SupCon encoder (+ proj_head, kept
# only so the saved state_dict's keys match; never used at inference) +
# the trained AttnPoolDocHead.
# ==============================================================================
print("\n" + "─" * 70)
print("  Loading Proposed Method checkpoint ...")

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hid, out):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hid), nn.BatchNorm1d(hid),
                                 nn.ReLU(inplace=True), nn.Linear(hid, out))
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

class MLPHead(nn.Module):
    """Same class as in the training script: Linear(in->in//2) -> ReLU ->
    Dropout -> Linear(in//2 -> num_labels)."""
    def __init__(self, in_dim, num_labels=2, dropout_prob=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.Linear(in_dim // 2, num_labels),
        )
    def forward(self, x):
        return self.net(x)

class SharedModel(nn.Module):
    def __init__(self, name, proj_hid, proj_dim, num_labels=2, cls_dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        hs = self.encoder.config.hidden_size
        self.proj_head = ProjectionHead(hs, proj_hid, proj_dim)
        self.cls_head = MLPHead(hs, num_labels=num_labels, dropout_prob=cls_dropout)   # sentence head (kept so strict load matches; unused here)
    def get_pooled(self, ids, mask):
        hidden = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        mask_expanded = mask.unsqueeze(-1).float()
        summed = (hidden * mask_expanded).sum(dim=1)
        counts = mask_expanded.sum(dim=1).clamp(min=1e-9)
        return summed / counts

class AttnPoolDocHead(nn.Module):
    """IDENTICAL architecture to
    deberta_v3_supcon_attnpool_dochead_english.py's AttnPoolDocHead: a
    small scoring network gives each sentence a learned weight (masked
    softmax over the document's own sentences); the document vector is
    their WEIGHTED SUM -> small MLP classifier -> 1 logit."""
    def __init__(self, in_dim, attn_hidden=256, dropout=0.3):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(in_dim, attn_hidden), nn.Tanh(), nn.Linear(attn_hidden, 1),
        )
        self.classifier = MLPHead(in_dim, num_labels=1, dropout_prob=dropout)   # MODIFIED: same MLPHead class as training
    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        pooled = (x * weights.unsqueeze(-1)).sum(dim=1)
        logit = self.classifier(pooled).squeeze(-1)
        return logit, pooled, weights

m1_tokenizer = AutoTokenizer.from_pretrained(f"{M1_DRIVE_PATH}/tokenizer_{NEW_SUFFIX}")
m1_model = SharedModel(M1_ENCODER_NAME, M1_PROJ_HID, M1_PROJ_DIM).to(DEVICE)
m1_ckpt = torch.load(f"{M1_DRIVE_PATH}/encoder_{NEW_SUFFIX}.pt", map_location=DEVICE, weights_only=False)
m1_model.load_state_dict(m1_ckpt["shared_model"], strict=True)
m1_model = m1_model.float().to(DEVICE).eval()
for p in m1_model.parameters():
    p.requires_grad = False
M1_HIDDEN = m1_model.encoder.config.hidden_size

attn_doc_head = AttnPoolDocHead(M1_HIDDEN, ATTN_HIDDEN, HEAD_DROPOUT).to(DEVICE)
_attn_ckpt = torch.load(f"{ATTNPOOL_HEAD_DRIVE_PATH}/attnpool_doc_head_{NEW_SUFFIX}.pt", map_location=DEVICE, weights_only=False)
attn_doc_head.load_state_dict(_attn_ckpt["attn_doc_head_state_dict"])
attn_doc_head = attn_doc_head.float().to(DEVICE).eval()
for p in attn_doc_head.parameters():
    p.requires_grad = False

with open(f"{ATTNPOOL_HEAD_DRIVE_PATH}/config_{NEW_SUFFIX}.json") as f:
    _attn_head_cfg = json.load(f)
THRESHOLDS_DOC[ATTN_KEY] = float(_attn_head_cfg["th_doc"])

print(f"  Proposed Method loaded. Encoder={M1_ENCODER_NAME}  hidden={M1_HIDDEN}  "
      f"threshold={THRESHOLDS_DOC[ATTN_KEY]:.2f}")

threshold_table = pd.DataFrame({
    "Method": METHOD_ORDER,
    "Threshold": [round(THRESHOLDS_DOC[m], 2) for m in METHOD_ORDER],
})
print("\n  Threshold reference table (used for every table/heatmap below):")
print(threshold_table.to_string(index=False))

# ==============================================================================
# 5. INFERENCE — one function per method family
# ==============================================================================
@torch.no_grad()
def predict_probs_doclevel_classifier(method_name, texts, batch_size=DOC_INFER_BATCH, desc_suffix=""):
    """Whole DOCUMENT as ONE sequence (MAX_LENGTH=512), straight to
    softmax — exactly like that backbone's own
    *_finetune_document_level_english.py training/testing scripts."""
    tok, mdl = CLASSIFIERS[method_name]
    probs = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(texts), batch_size), total=n_batches,
                  desc=f"  {method_name} {desc_suffix}", leave=False):
        batch = texts[i:i + batch_size]
        enc = tok(batch, truncation=True, max_length=DOC_MAX_LEN, padding=True,
                  return_tensors="pt").to(DEVICE)
        logits = mdl(**enc).logits
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs) if probs else np.array([])

@torch.no_grad()
def encode_sentences_m1(sent_list, batch_size=32):
    if not sent_list:
        return np.zeros((0, M1_HIDDEN), dtype=np.float32)
    embs = []
    for i in range(0, len(sent_list), batch_size):
        batch = sent_list[i:i + batch_size]
        enc = m1_tokenizer(batch, truncation=True, max_length=M1_MAX_LEN, padding=True,
                           return_tensors="pt").to(DEVICE)
        pooled = m1_model.get_pooled(enc["input_ids"], enc["attention_mask"])
        embs.append(pooled.cpu().numpy().astype(np.float32))
    return np.concatenate(embs, axis=0)

@torch.no_grad()
def predict_prob_attnpool_doc(raw_text):
    """Attention-Pooling (Doc) — frozen SupCon sentence embeddings, pooled
    by the trained head's learned per-sentence attention weights (weighted
    sum) instead of an unweighted mean."""
    sents = doc_to_sentences_for_pooling(raw_text)
    if not sents:
        return 0.5   # no usable sentence found -> neutral, no-evidence baseline
    sent_emb = encode_sentences_m1(sents)
    x = torch.from_numpy(sent_emb).unsqueeze(0).to(DEVICE)
    mask = torch.ones(1, sent_emb.shape[0], dtype=torch.bool, device=DEVICE)
    logit, _, _ = attn_doc_head(x, mask)
    return float(torch.sigmoid(logit).item())

# ==============================================================================
# 6. دالة التقييم الكاملة لعمود نص واحد (ONE attack column, ALL 7 methods)
# ==============================================================================
def evaluate_text_column(texts, batch_desc=""):
    valid_mask = np.array([not str(t).startswith("[ATTACK_FAILED") for t in texts])
    texts_norm_doc = [normalize_english_text(t) for t in texts]

    out = {}
    for m in CLASSIFIER_PATHS:
        out[m] = predict_probs_doclevel_classifier(m, texts_norm_doc, desc_suffix=batch_desc)

    attn_probs = []
    for t in tqdm(texts, desc=f"  {ATTN_KEY} {batch_desc}", leave=False):
        attn_probs.append(predict_prob_attnpool_doc(t))
    out[ATTN_KEY] = np.array(attn_probs)

    out["valid_mask"] = valid_mask
    return out

# ==============================================================================
# 7. رفع/تحديد ملف الهجمات
# ==============================================================================
uploaded = files.upload()   # اختر kufa_ai_generated_attacked.xlsx
ATTACKS_FILE = list(uploaded.keys())[0]
print("تم تحميل ملف الهجمات:", ATTACKS_FILE)

df_att = pd.read_excel(ATTACKS_FILE)
print("شكل ملف الهجمات:", df_att.shape)
print("الأعمدة:", list(df_att.columns))

if EVAL_SAMPLE_SIZE is not None:
    df_att = df_att.sample(n=min(EVAL_SAMPLE_SIZE, len(df_att)), random_state=42).reset_index(drop=True)
print("عدد الصفوف اللي راح نقيّمها:", len(df_att))

# ==============================================================================
# 8. الحلقة الرئيسية — كل عمود هجوم × السبع طرق
# ==============================================================================
results_long = []
per_row_probs = {}

for col, display_name in ATTACK_COLUMNS.items():
    if col not in df_att.columns:
        print(f"⚠ العمود {col} غير موجود بالملف — تجاوزته.")
        continue
    raw_na_mask = df_att[col].isna().to_numpy()   # e.g. "stealthwriter humanizer" is only
    # partially filled — NaN rows must be excluded, not evaluated as the
    # literal string "nan" (see ATTACK_COLUMNS note above).
    texts = df_att[col].astype(str).tolist()
    out = evaluate_text_column(texts, batch_desc=f"({display_name})")
    out["valid_mask"] = out["valid_mask"] & ~raw_na_mask
    per_row_probs[col] = out
    vm = out["valid_mask"]
    n_valid = int(vm.sum())
    if raw_na_mask.any():
        print(f"    ({int(raw_na_mask.sum())} empty/NaN rows in '{col}' excluded from evaluation)")

    for method in METHOD_ORDER:
        probs = out[method][vm]
        preds = (probs >= THRESHOLDS_DOC[method]).astype(int)
        det_rate = float(preds.mean()) if n_valid > 0 else np.nan
        results_long.append({
            "attack_col": col, "attack_name": display_name, "method": method,
            "threshold": round(THRESHOLDS_DOC[method], 2),
            "n_valid": n_valid, "detection_rate": round(det_rate, 4),
            "mean_prob": round(float(probs.mean()), 4) if n_valid > 0 else np.nan,
        })
    print(f"  ✓ {display_name:<24} n_valid={n_valid}")

results_long_df = pd.DataFrame(results_long)
print("\nنتائج التقييم (long format):")
print(results_long_df.to_string(index=False))

# ==============================================================================
# 9. جدول Detection Rate (Attack × Method) — 7 أعمدة
# ==============================================================================
det_matrix = results_long_df.pivot(index="attack_name", columns="method", values="detection_rate")
det_matrix = det_matrix.reindex(index=[ATTACK_COLUMNS[c] for c in ATTACK_COLUMNS if c in df_att.columns],
                                 columns=METHOD_ORDER)
print("Detection Rate matrix (Attack × Method):")
print(det_matrix.to_string())

# ==============================================================================
# 10. جدول Accuracy Drop = Baseline − Attack — 7 أعمدة
# ==============================================================================
baseline_row = det_matrix.loc["Baseline (No Attack)"]
drop_matrix = det_matrix.drop(index="Baseline (No Attack)").apply(lambda row: baseline_row - row, axis=1)
drop_matrix = drop_matrix.round(4)
print("\nAccuracy / Detection-Rate Drop matrix (Baseline − Attack):")
print(drop_matrix.to_string())

# ==============================================================================
# 11. ترتيب الهجمات حسب متوسط قوة التخريب (عبر السبع طرق)
# ==============================================================================
attack_ranking = drop_matrix.mean(axis=1).sort_values(ascending=False).reset_index()
attack_ranking.columns = ["attack_name", "avg_drop_across_methods"]
attack_ranking["avg_drop_across_methods"] = attack_ranking["avg_drop_across_methods"].round(4)
print("\nترتيب الهجمات حسب متوسط الانخفاض بكل الطرق:")
print(attack_ranking.to_string(index=False))

# ==============================================================================
# FIGURE 1 — Heatmap: Detection Rate (Q1 sequential BLUE, dynamic text color)
# ==============================================================================
fig1, ax1 = plt.subplots(figsize=(2.1 * len(METHOD_ORDER), max(6, 0.5 * len(det_matrix) + 2)))
det_f = det_matrix.astype(float)
sns.heatmap(det_f, annot=False, cmap=BLUE_ONLY_CMAP, vmin=0, vmax=1,
            linewidths=0.8, linecolor="white",
            cbar_kws={"label": "Detection Rate", "shrink": 0.85}, ax=ax1, square=False)
heatmap_dynamic_text(ax1, det_f, 0, 1, BLUE_ONLY_CMAP, fontsize=10.5)
style_axis(ax1, "Detection Rate by Attack × Method", "Method", "Attack Type")
ax1.tick_params(axis="x", rotation=20); ax1.tick_params(axis="y", rotation=0)
plt.tight_layout()
p1 = os.path.join(OUTPUT_DIR, "heatmap_detection_rate_by_attack.png")
plt.savefig(p1, bbox_inches="tight"); plt.show()
print("Saved ->", p1)

# ==============================================================================
# FIGURE 2 — Heatmap: Accuracy / Detection-Rate Drop (Q1 sequential RED)
# ==============================================================================
fig2, ax2 = plt.subplots(figsize=(2.1 * len(METHOD_ORDER), max(6, 0.5 * len(drop_matrix) + 2)))
vmax_drop = max(0.05, float(drop_matrix.values.max()))
drop_f = drop_matrix.astype(float)
sns.heatmap(drop_f, annot=False, cmap=RED_ONLY_CMAP, vmin=0, vmax=vmax_drop,
            linewidths=0.8, linecolor="white",
            cbar_kws={"label": "Detection-Rate Drop", "shrink": 0.85}, ax=ax2, square=False)
heatmap_dynamic_text(ax2, drop_f, 0, vmax_drop, RED_ONLY_CMAP, fontsize=10.5)
style_axis(ax2, "Robustness Degradation — Detection-Rate Drop vs Baseline", "Method", "Attack Type")
ax2.tick_params(axis="x", rotation=20); ax2.tick_params(axis="y", rotation=0)
plt.tight_layout()
p2 = os.path.join(OUTPUT_DIR, "heatmap_accuracy_drop_by_attack.png")
plt.savefig(p2, bbox_inches="tight"); plt.show()
print("Saved ->", p2)

# ==============================================================================
# FIGURE 3 — Bar chart: ترتيب الهجمات (Q1 sequential RED, scaled by value)
# ==============================================================================
fig3, ax3 = plt.subplots(figsize=(10, max(6, 0.4 * len(attack_ranking) + 2)))
norm_vals = attack_ranking["avg_drop_across_methods"] / max(attack_ranking["avg_drop_across_methods"].max(), 1e-6)
colors3 = [RED_ONLY_CMAP(0.25 + 0.65 * v) for v in norm_vals]
bars3 = ax3.barh(attack_ranking["attack_name"], attack_ranking["avg_drop_across_methods"],
                 color=colors3, edgecolor="white", linewidth=1.2, height=0.62, zorder=3)
ax3.invert_yaxis()
style_axis(ax3, "Attack Strength Ranking (Most Damaging First)",
          "Average Detection-Rate Drop (across 7 methods)")
for b, v in zip(bars3, attack_ranking["avg_drop_across_methods"]):
    ax3.text(v + 0.005, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center",
              fontsize=9, fontweight="bold", color=AXIS_COLOR)
plt.tight_layout()
p3 = os.path.join(OUTPUT_DIR, "attack_strength_ranking.png")
plt.savefig(p3, bbox_inches="tight"); plt.show()
print("Saved ->", p3)

# ==============================================================================
# FIGURE 4 — Grouped bar chart: Detection Rate per Attack — all 7 methods
# ==============================================================================
fig4, ax4 = plt.subplots(figsize=(max(16, 1.0 * len(det_matrix) + 4), 6.5))
attacks_x = det_matrix.index.tolist()
x = np.arange(len(attacks_x))
n_meth = len(METHOD_ORDER)
bw = 0.86 / n_meth
for i, (m, color) in enumerate(zip(METHOD_ORDER, METHOD_COLORS)):
    vals = det_matrix[m].values
    offs = (i - (n_meth - 1) / 2) * bw
    ax4.bar(x + offs, vals, width=bw * 0.88, color=color, edgecolor="white", linewidth=1.0, label=m, zorder=3)
ax4.axhline(0.5, color=AXIS_COLOR, lw=1.2, ls="--", alpha=0.6, zorder=2)
ax4.set_xticks(x); ax4.set_xticklabels(attacks_x, rotation=30, ha="right")
ax4.set_ylim(0, 1.08)
style_axis(ax4, "Detection Rate per Attack — All Seven Methods", ylabel="Detection Rate")
ax4.legend(fontsize=9, ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.30))
plt.tight_layout()
p4 = os.path.join(OUTPUT_DIR, "detection_rate_grouped_bars.png")
plt.savefig(p4, bbox_inches="tight"); plt.show()
print("Saved ->", p4)

# ==============================================================================
# FIGURE 5 — Degradation line curve — all 7 methods
# ==============================================================================
fig5, ax5 = plt.subplots(figsize=(12, 6.2))
order = attack_ranking["attack_name"].tolist()
plot_order = ["Baseline (No Attack)"] + order
for m, color in zip(METHOD_ORDER, METHOD_COLORS):
    vals = det_matrix.loc[plot_order, m].values
    ax5.plot(range(len(plot_order)), vals, marker="o", lw=2.4, ms=6, color=color, label=m, zorder=3)
ax5.set_xticks(range(len(plot_order)))
ax5.set_xticklabels(plot_order, rotation=35, ha="right", fontsize=8)
ax5.set_ylim(0, 1.08)
style_axis(ax5, "Detection-Rate Degradation Curve (sorted by attack strength)", ylabel="Detection Rate")
ax5.legend(fontsize=9, ncol=4)
plt.tight_layout()
p5 = os.path.join(OUTPUT_DIR, "degradation_curve.png")
plt.savefig(p5, bbox_inches="tight"); plt.show()
print("Saved ->", p5)

# ==============================================================================
# FIGURE 6 — Domain heatmap لأقوى هجومين (Q1 sequential BLUE)
# ==============================================================================
domain_heatmaps_paths = []
if "domain" in df_att.columns:
    top2_attacks = attack_ranking.head(2)["attack_name"].tolist()
    name_to_col = {v: k for k, v in ATTACK_COLUMNS.items()}

    for atk_name in top2_attacks:
        col = name_to_col[atk_name]
        out = per_row_probs[col]
        vm = out["valid_mask"]
        domains = df_att["domain"].astype(str).values[vm]

        rows = []
        for dm in sorted(set(domains)):
            m = domains == dm
            if m.sum() == 0: continue
            row = {"domain": dm, "n": int(m.sum())}
            for method in METHOD_ORDER:
                probs = out[method][vm][m]
                preds = (probs >= THRESHOLDS_DOC[method]).astype(int)
                row[method] = round(float(preds.mean()), 4)
            rows.append(row)
        dom_df = pd.DataFrame(rows).set_index("domain")[METHOD_ORDER]

        figd, axd = plt.subplots(figsize=(2.1 * len(METHOD_ORDER), max(5, 0.5 * len(dom_df) + 2)))
        dom_f = dom_df.astype(float)
        sns.heatmap(dom_f, annot=False, cmap=BLUE_ONLY_CMAP, vmin=0, vmax=1,
                    linewidths=0.8, linecolor="white",
                    cbar_kws={"label": "Detection Rate", "shrink": 0.85}, ax=axd, square=False)
        heatmap_dynamic_text(axd, dom_f, 0, 1, BLUE_ONLY_CMAP, fontsize=10.5)
        style_axis(axd, f"Detection Rate by Domain × Method\nAttack: {atk_name}", "Method", "Domain")
        axd.tick_params(axis="x", rotation=20)
        plt.tight_layout()
        safe_name = atk_name.replace(" ", "_").replace("/", "-")
        pdpath = os.path.join(OUTPUT_DIR, f"domain_heatmap_{safe_name}.png")
        plt.savefig(pdpath, bbox_inches="tight"); plt.show()
        domain_heatmaps_paths.append(pdpath)
        print("Saved ->", pdpath)
else:
    print("⚠ عمود 'domain' غير موجود — تجاوزت رسومات الـ domain heatmap.")

# ==============================================================================
# FIGURE 7 — Probability shift: Baseline vs Strongest Attack — all 7 methods
# ==============================================================================
strongest_attack_name = attack_ranking.iloc[0]["attack_name"]
strongest_col = {v: k for k, v in ATTACK_COLUMNS.items()}[strongest_attack_name]

fig7, axes7 = plt.subplots(1, len(METHOD_ORDER), figsize=(4.0 * len(METHOD_ORDER), 4.6))
fig7.suptitle(f"P(AI) Shift — Baseline vs Strongest Attack ({strongest_attack_name})",
             fontsize=14, fontweight="bold")
base_out = per_row_probs["text_original"]
atk_out  = per_row_probs[strongest_col]
for i, (ax, m) in enumerate(zip(axes7, METHOD_ORDER)):
    bvm = base_out["valid_mask"]; avm = atk_out["valid_mask"]
    ax.hist(base_out[m][bvm], bins=25, alpha=0.65, color=BASELINE_COLOR, label="Baseline", density=True, zorder=3)
    ax.hist(atk_out[m][avm], bins=25, alpha=0.65, color=ATTACK_COLOR, label=strongest_attack_name, density=True, zorder=3)
    ax.axvline(THRESHOLDS_DOC[m], color=AXIS_COLOR, lw=1.6, ls="--", zorder=4)
    style_axis(ax, m, "P(AI)")
    ax.legend(fontsize=7)
    add_panel_label(ax, chr(65 + i))
plt.tight_layout()
p7 = os.path.join(OUTPUT_DIR, "probability_shift_strongest_attack.png")
plt.savefig(p7, bbox_inches="tight"); plt.show()
print("Saved ->", p7)

# ==============================================================================
# حفظ الجداول والرسومات
# ==============================================================================
results_long_df.to_csv(os.path.join(OUTPUT_DIR, "detection_rate_long.csv"), index=False)
det_matrix.to_csv(os.path.join(OUTPUT_DIR, "detection_rate_matrix.csv"))
drop_matrix.to_csv(os.path.join(OUTPUT_DIR, "accuracy_drop_matrix.csv"))
attack_ranking.to_csv(os.path.join(OUTPUT_DIR, "attack_strength_ranking.csv"), index=False)
threshold_table.to_csv(os.path.join(OUTPUT_DIR, "threshold_reference_table.csv"), index=False)

for f in os.listdir(OUTPUT_DIR):
    shutil.copy(os.path.join(OUTPUT_DIR, f), DRIVE_EVAL_OUT)

print(f"✓ كل الجداول والرسومات انحفظت بـ:\n  محلي: {OUTPUT_DIR}\n  Drive: {DRIVE_EVAL_OUT}")
print("\nملفات محفوظة:", os.listdir(DRIVE_EVAL_OUT))

# ==============================================================================
# ملخص نهائي
# ==============================================================================
print("=" * 70)
print("  ROBUSTNESS EVALUATION — FINAL SUMMARY")
print("  (6 document-level classifiers + Proposed Method)")
print("=" * 70)
print(f"  Attacks evaluated   : {len(det_matrix) - 1}  (+ baseline)")
print(f"  Rows evaluated      : {len(df_att)}")
print()
print("  Baseline detection rate (no attack):")
print(det_matrix.loc["Baseline (No Attack)"].to_string())
print()
print(f"  Most damaging attack : {attack_ranking.iloc[0]['attack_name']} "
      f"(avg drop = {attack_ranking.iloc[0]['avg_drop_across_methods']:.4f})")
print(f"  Least damaging attack: {attack_ranking.iloc[-1]['attack_name']} "
      f"(avg drop = {attack_ranking.iloc[-1]['avg_drop_across_methods']:.4f})")
print()
most_robust_method = drop_matrix.mean(axis=0).idxmin()
least_robust_method = drop_matrix.mean(axis=0).idxmax()
print(f"  Most robust method overall  : {most_robust_method} "
      f"(avg drop = {drop_matrix.mean(axis=0)[most_robust_method]:.4f})")
print(f"  Least robust method overall : {least_robust_method} "
      f"(avg drop = {drop_matrix.mean(axis=0)[least_robust_method]:.4f})")
print("=" * 70)